# Exercise 1.2 — Find Duplicates in a CSV
### *One person, three "different" email addresses*

Anyone who has managed a mailing list, a client database, or an event RSVP sheet knows this pain: the same person shows up multiple times because someone typed `John.Tan@Gmail.com` and someone else typed `  john.tan@gmail.com `. To a human skimming a spreadsheet, these look "probably the same." To a computer doing a straight comparison, they're totally different strings — unless you teach it what "the same" actually means.

**Why learn this instead of just uploading the CSV to an AI chatbot and asking it to dedupe?**

You can do that too! But once you understand *why* `"A@b.com" != " a@b.com "` to a computer, you'll immediately understand why so many "AI cleaned my data" results are subtly wrong — trailing spaces, inconsistent casing, and near-duplicates that don't quite match. You'll know what to check for, in *any* tool.

## What you'll practise
- **Lists** — collecting rows of data
- **Dictionaries** — looking things up by key (like "have I seen this email before?")
- **Conditionals** — deciding what counts as a duplicate

## The scenario
You've been handed `messy_contacts.csv` — a real (if fictional) messy contact list with ~52 rows, some of which are the same person entered more than once, with slightly different formatting each time. Let's find them.

## Step 1: Read the CSV and look at it

In [1]:
import csv

with open("messy_contacts.csv", newline="") as f:
    reader = csv.DictReader(f)
    contacts = list(reader)

print(f"Loaded {len(contacts)} rows")
print("First 5 rows:")
for row in contacts[:5]:
    print(row)


Loaded 52 rows
First 5 rows:
{'name': 'Nur Kumar', 'email': 'nur.kumar@outlook.com', 'phone': '98444775'}
{'name': 'Ben Tan', 'email': 'ben.tan@hotmail.com', 'phone': '93437776'}
{'name': 'Marcus Tan', 'email': 'marcus.tan@company.com', 'phone': '95688232'}
{'name': 'Jason Lim', 'email': 'jason.lim@company.com', 'phone': '96536500'}
{'name': 'Michelle Foo', 'email': 'michelle.foo@outlook.com', 'phone': '9-8878-414'}


**What just happened?**

- `csv.DictReader` reads each row of the CSV into a **dictionary** — a set of `key: value` pairs, like `{"name": "Alex Tan", "email": "alex.tan@gmail.com", "phone": "91234567"}`. This is much easier to work with than a plain list of columns, because you can say `row["email"]` instead of remembering "email is column 2."
- `list(reader)` turns the whole thing into a **list of dictionaries** — one dictionary per contact.

## Step 2: Why a naive check fails

Let's try the "obvious" approach first — just comparing emails directly — and see it fail.

In [2]:
seen_emails = []
naive_duplicates = []

for row in contacts:
    email = row["email"]
    if email in seen_emails:
        naive_duplicates.append(row)
    else:
        seen_emails.append(email)

print(f"Naive approach found {len(naive_duplicates)} duplicates.")
print("(Suspiciously low, right? Let's see why.)")


Naive approach found 4 duplicates.
(Suspiciously low, right? Let's see why.)


Because some emails have different capitalisation or stray spaces (`"  Alex.Tan@GMAIL.com "` vs `"alex.tan@gmail.com"`), Python's plain `==` comparison treats them as completely different strings. We need to **normalise** the data before comparing it — this is one of the most important ideas in all of data cleaning.

## Step 3: Normalise, then compare

In [3]:
def normalise_email(email):
    # .strip() removes leading/trailing whitespace
    # .lower() makes everything lowercase so casing doesn't matter
    return email.strip().lower()

# quick test
print(repr(normalise_email("  Alex.Tan@GMAIL.com ")))
print(repr(normalise_email("alex.tan@gmail.com")))
print(normalise_email("  Alex.Tan@GMAIL.com ") == normalise_email("alex.tan@gmail.com"))


'alex.tan@gmail.com'
'alex.tan@gmail.com'
True


`repr()` here just shows us the *exact* string, including hidden spaces, so we can see what `.strip()` removed. Now let's re-run the duplicate check using the normalised version.

In [4]:
seen = {}          # dictionary: normalised email -> the first row we saw with that email
duplicates = []     # list of rows that turned out to be repeats

for row in contacts:
    key = normalise_email(row["email"])

    if key in seen:
        duplicates.append(row)
    else:
        seen[key] = row

print(f"Found {len(duplicates)} duplicate rows out of {len(contacts)} total.")
print(f"That means there are really only {len(seen)} unique people in this list.")


Found 12 duplicate rows out of 52 total.
That means there are really only 40 unique people in this list.


**Why a dictionary here?** Checking `if key in seen` on a dictionary is fast and, more importantly, reads exactly like the plain-English question we're asking: *"Have I seen this email before?"* That's the whole trick — a dictionary is Python's built-in way of asking "do I already know about this thing?" instantly.

## Step 4: Show the actual duplicate pairs (so a human can verify)

In [5]:
seen = {}
duplicate_pairs = []

for row in contacts:
    key = normalise_email(row["email"])
    if key in seen:
        duplicate_pairs.append((seen[key], row))
    else:
        seen[key] = row

for original, dupe in duplicate_pairs:
    print("ORIGINAL:", original)
    print("DUPLICATE:", dupe)
    print("-" * 60)


ORIGINAL: {'name': 'Siti Yeo', 'email': 'siti.yeo@hotmail.com', 'phone': '97110140'}
DUPLICATE: {'name': 'Siti Yeo', 'email': 'siti.yeo@hotmail.com', 'phone': '9-7110-140'}
------------------------------------------------------------
ORIGINAL: {'name': 'Kevin Chua', 'email': 'kevin.chua@company.com', 'phone': '97095538'}
DUPLICATE: {'name': 'Kevin Chua', 'email': 'KEVIN.CHUA@COMPANY.COM', 'phone': '97095538'}
------------------------------------------------------------
ORIGINAL: {'name': 'Tom Rahman', 'email': '  tom.rahman@outlook.com ', 'phone': '94819238'}
DUPLICATE: {'name': 'Tom Rahman', 'email': 'tom.rahman@outlook.com', 'phone': '94819238'}
------------------------------------------------------------
ORIGINAL: {'name': 'Ben Tan', 'email': 'ben.tan@hotmail.com', 'phone': '93437776'}
DUPLICATE: {'name': 'Ben Tan', 'email': '  ben.tan@hotmail.com ', 'phone': '93437776'}
------------------------------------------------------------
ORIGINAL: {'name': 'Siti Tan', 'email': 'siti.tan@ou

Notice: never blindly auto-delete "duplicates" in real data work. Always print both versions side-by-side first, like we just did, so a human can confirm they really are the same person before anything gets removed.

## 🎉 You did it

You just learned the core skill behind every "deduplicate my list" task: **decide what "the same" means, normalise your data to that standard, then compare.** This exact idea — normalise before you compare — applies to phone numbers, company names, addresses, product SKUs... basically any messy real-world text.

---

## 🧪 Try it yourself

1. The sample data also has some duplicate *phone numbers* with different formatting (e.g. `9-1234-567` vs `91234567`). Write a `normalise_phone()` function that strips out dashes and spaces, then use it to find phone-based duplicates too.
2. Some names appear in different casing (`JOHN TAN` vs `John Tan`). Extend the duplicate check to also catch same-name-different-email cases — are these the same person or different people? What would you need to know to decide?
3. Write out a *clean* version of the CSV containing only unique contacts (keep the first occurrence of each), using `csv.DictWriter`.

## 💡 Where this goes next
This "normalise, then compare" pattern is exactly what customer databases, email marketing tools, and CRMs do under the hood when they say "we automatically merge duplicate contacts." Now you know it's not magic — it's a dictionary and a `.strip().lower()`.